In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

1. Loading Datasets

In [ ]:
print("Initiating Regression Model Pipeline...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

In [ ]:
def preprocess_features(df):
    """
    Feature engineering and missing value handling for cognitive performance dataset.
    Optimized for CatBoost Regressor.
    """
    # Handling missing values for nap duration
    df['sekerleme_suresi_dk'] = df['sekerleme_suresi_dk'].fillna(0)
    df['is_napping'] = (df['sekerleme_suresi_dk'] > 0).astype(int)
    
    # --- Feature Engineering Pipeline ---
    
    # 1. Sleep Efficiency (REM + Deep Sleep normalized by Sleep Latency)
    df['sleep_efficiency_ratio'] = (df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']) / (df['uykuya_dalma_suresi_dk'] + 5)
    
    # 2. Metabolic Efficiency Index
    df['metabolic_index'] = df['gunluk_adim_sayisi'] / (df['dinlenik_nabiz_bpm'] * df['vucut_kitle_indeksi'] + 1)
    
    # 3. Mental Fatigue (Interaction between screen time and stress level)
    df['mental_fatigue_index'] = df['stres_skoru'] * (df['uyku_oncesi_ekran_suresi_dk'] / 60)
    
    # 4. Sleep Debt Impact (Weighted by age)
    df['sleep_debt_impact'] = np.abs(df['hafta_sonu_uyku_farki_saat']) * df['yas']
    
    # 5. Stimulant Burden (Interaction between caffeine and screen exposure)
    df['stimulant_burden'] = df['uyku_oncesi_kafein_mg'] * df['uyku_oncesi_ekran_suresi_dk']
    
    # 6. Occupational Stress Load
    df['occupational_stress_load'] = df['stres_skoru'] * df['gunluk_calisma_saati']

    # Categorical Encoding preparation
    cat_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
    for col in cat_cols:
        df[col] = df[col].astype(str)
        
    return df, cat_cols

Preprocessing execution

In [ ]:
train, cat_features = preprocess_features(train)
test, _ = preprocess_features(test)

In [ ]:
X = train.drop(['id', 'bilissel_performans_skoru'], axis=1)
y = train['bilissel_performans_skoru']
X_test = test.drop(['id'], axis=1)

2. Hyperparameters (Optimized for Generalization)

In [ ]:
cb_params = {
    'iterations': 6000,
    'learning_rate': 0.012,
    'depth': 7,
    'l2_leaf_reg': 15,
    'random_strength': 1.5,
    'bagging_temperature': 1,
    'eval_metric': 'RMSE',
    'random_seed': 42,
    'early_stopping_rounds': 400,
    'verbose': 500,
    'task_type': 'CPU',  # Can be switched to 'GPU' for supported environments
    'cat_features': cat_features
}

3. Cross-Validation Loop (5-Fold)

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))

In [ ]:
print("\nStarting 5-Fold Cross-Validation...")

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(**cb_params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    
    oof_predictions[val_idx] = model.predict(X_val)
    test_predictions += model.predict(X_test) / kf.n_splits
    
    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_predictions[val_idx]))
    print(f"Fold {fold+1} Validation RMSE: {fold_rmse:.5f}")

Final Performance Evaluation

In [ ]:
total_rmse = np.sqrt(mean_squared_error(y, oof_predictions))
print(f"\nAggregate OOF RMSE Score: {total_rmse:.5f}")

4. Submission Export

In [ ]:
filename = f'submission_optimized_v9_rmse_{total_rmse:.5f}.csv'
submission = pd.DataFrame({
    'id': test['id'], 
    'bilissel_performans_skoru': test_predictions
})
submission.to_csv(filename, index=False)

In [ ]:
print(f"Execution successful. Output saved to: {filename}")